# Listwise Learning to Rank con AdaRank — consultas LOINC

Notebook de demostración. Toda la lógica está en `src/`:

| Script | Qué hace |
|---|---|
| `src/queries.py` | Especificación de las consultas y regla de relevancia 0–3 |
| `src/label.py` | Aplica la regla a los 67 documentos del Excel → `data/relevance_labels.csv` |
| `src/build_dataset.py` | Features f1–f5 + LETOR → `data/features.csv`, `data/train.txt` |
| `src/evaluate.py` | Evaluación leave-one-query-out de AdaRank frente a baselines |

La implementación de AdaRank (`adarank.py`, `metrics.py`, `utils.py`) está adaptada de
[rueycheng/AdaRank](https://github.com/rueycheng/AdaRank).

In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
from sklearn.datasets import load_svmlight_file

from adarank import AdaRank
from metrics import NDCGScorer
import build_dataset
from build_dataset import FEATURES

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

## 1. Construir el training set

Regenera etiquetas, features y archivos LETOR, y ejecuta las comprobaciones
(201 pares, 67 por consulta, mismos códigos en cada consulta, etiquetas 0–3, features finitas).

In [ ]:
build_dataset.main()

In [ ]:
labels = pd.read_csv('data/relevance_labels.csv', dtype={'loinc_num': str})
official = labels[labels.official]
official.pivot_table(index='query', columns='relevance', values='loinc_num',
                     aggfunc='count', fill_value=0, sort=False)

### Documentos con relevancia > 0 y su justificación

In [ ]:
(official[official.relevance > 0]
 .sort_values(['qid', 'relevance'], ascending=[True, False])
 [['query', 'loinc_num', 'long_common_name', 'relevance', 'rationale']])

## 2. Features (calculadas sin mirar la etiqueta)

In [ ]:
feats = pd.read_csv('data/features.csv', dtype={'loinc_num': str})
data = labels.merge(feats, on=['qid', 'query', 'loinc_num'])
data[data.qid == 3].sort_values('relevance', ascending=False)[
    ['long_common_name', 'relevance'] + FEATURES].head(8)

### Formato LETOR (`data/train.txt`)

In [ ]:
print(''.join(open('data/train.txt', encoding='utf-8').readlines()[:5]))

## 3. Entrenamiento y evaluación leave-one-query-out

Con 3 consultas, cada fold entrena con 2 y evalúa en la tercera. No se dividen
documentos de una misma consulta entre train y test: la unidad Listwise es la lista.

In [ ]:
X, y, qid = load_svmlight_file('data/train.txt', query_id=True, n_features=len(FEATURES))
rows = []
for test_q in np.unique(qid):
    tr, te = qid != test_q, qid == test_q
    model = AdaRank(max_iter=100, estop=10, scorer=NDCGScorer(k=10)).fit(X[tr], y[tr], qid[tr])
    for name, pred in [('AdaRank', model.predict(X[te], qid[te])),
                       ('baseline f1 (TF-IDF)', X[te][:, 0].toarray().ravel())]:
        rows.append({'test_qid': int(test_q), 'sistema': name,
                     **{f'NDCG@{k}': NDCGScorer(k=k)(y[te], pred, qid[te])[0] for k in (1, 3, 5, 10)}})
loqo = pd.DataFrame(rows)
loqo

In [ ]:
loqo.groupby('sistema').mean(numeric_only=True).drop(columns='test_qid')

## 4. Modelo final (3 consultas) y ranking por consulta

In [ ]:
model = AdaRank(max_iter=100, estop=10, scorer=NDCGScorer(k=10)).fit(X, y, qid)
print(dict(zip(FEATURES, model.coef_.round(4))))

off = data[data.official].copy()   # mismo orden que data/train.txt
off['score'] = model.predict(X, qid)
for q, g in off.groupby('query', sort=False):
    print('\n==', q)
    print(g.sort_values('score', ascending=False)[['long_common_name', 'relevance', 'score']]
          .head(8).to_string(index=False))

## 5. Consulta no vista en el entrenamiento

El modelo entrenado con las 3 consultas oficiales se aplica a `cholesterol in plasma`
(qid 5, parte opcional). Se puntúan los **67 documentos únicos** (el notebook anterior
repetía cada documento 3 veces). La etiqueta solo se usa para evaluar.

In [ ]:
new = data[data.qid == 5].copy()
new['score'] = new[FEATURES].to_numpy() @ model.coef_   # mismo cálculo que model.predict
assert new.loinc_num.is_unique and len(new) == 67
print('NDCG@10 =', NDCGScorer(k=10)(new.relevance.to_numpy(), new.score.to_numpy(), np.full(len(new), 5))[0])
new.sort_values('score', ascending=False)[['long_common_name', 'relevance', 'score']].head(8)

## 6. Dataset con más consultas (opcional)

`data/train_extended.txt` añade 3 consultas (calcium in serum, cholesterol in plasma,
chloride in blood) etiquetadas con la misma regla. Pasamos de 3 a 6 folds en LOQO.

In [ ]:
import subprocess
print(subprocess.run([sys.executable, 'src/evaluate.py', '--extended'], capture_output=True, text=True).stdout)